# 예제 05. 이미지 학습에서 만나는 오류
빅데이터프로그래밍 · 7주차

아래 셀들은 **일부러 오류가 나도록** 만들어져 있습니다.
MNIST 실습에서 학생들이 실제로 만나는 세 가지입니다.


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

test_set = datasets.MNIST("./data", train=False, download=True,
                          transform=transforms.ToTensor())
loader = DataLoader(test_set, batch_size=32)
images, labels = next(iter(loader))
print("images:", images.shape, "labels:", labels.shape)


## 1. 이미지 shape 오류 — Flatten을 안 했다


In [ ]:
model = nn.Sequential(nn.Linear(784, 10))
try:
    model(images)
except RuntimeError as err:
    print("RuntimeError:", err)


In [ ]:
# 해결 A — 모델에 Flatten 넣기
fixed = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
print("해결 A:", fixed(images).shape)

# 해결 B — 넣기 전에 펼치기
print("해결 B:", model(images.view(images.shape[0], -1)).shape)


## 2. batch 차원까지 합쳐 버렸다
오류가 나지 않고 조용히 이상한 결과가 나옵니다 — 더 위험합니다.


In [ ]:
wrong = images.view(-1, 784)      # 우연히 맞아 보이지만
print("모양:", wrong.shape, "← batch 32가 유지됐는지 확인")

really_wrong = images.reshape(1, -1)
print("완전히 뭉친 경우:", really_wrong.shape)


## 3. 출력 클래스 수 오류


In [ ]:
bad_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 5))    # 10이어야 하는데 5
loss_fn = nn.CrossEntropyLoss()

try:
    loss_fn(bad_model(images), labels)
except Exception as err:
    print(type(err).__name__, ":", err)


In [ ]:
# 해결: 출력 뉴런 수 = 클래스 수
good_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10))
print("손실:", loss_fn(good_model(images), labels).item())


## 4. 정답 형태 오류 — one-hot을 주면 안 됩니다
`CrossEntropyLoss` 는 클래스 번호(정수)를 받습니다.


In [ ]:
onehot = torch.zeros(32, 10)
onehot[range(32), labels] = 1.0

try:
    loss_fn(good_model(images), onehot.argmax(dim=1).float())
except Exception as err:
    print(type(err).__name__, ":", err)


In [ ]:
# 해결: long 타입 클래스 번호
print("정답 dtype:", labels.dtype)
print("손실:", loss_fn(good_model(images), labels).item())


## 5. CPU / GPU 장치 불일치
4주차에서 본 오류가 모델과 데이터 사이에서 다시 나타납니다.


In [ ]:
if torch.cuda.is_available():
    gpu_model = nn.Sequential(nn.Flatten(), nn.Linear(784, 10)).to("cuda")
    try:
        gpu_model(images)          # images 는 CPU 에 있음
    except RuntimeError as err:
        print("RuntimeError:", err)
else:
    print("GPU 런타임에서 실행하세요")


In [ ]:
# 해결: 둘 다 같은 장치로
if torch.cuda.is_available():
    print("해결:", gpu_model(images.to("cuda")).shape)


## 6. 예측 결과를 그리려는데 나는 오류
GPU Tensor는 matplotlib에 바로 넘길 수 없습니다.


In [ ]:
if torch.cuda.is_available():
    import matplotlib.pyplot as plt
    g = images[0].to("cuda")
    try:
        plt.imshow(g.squeeze(), cmap="gray")
    except TypeError as err:
        print("TypeError:", err)
    plt.close()
    print("해결: .cpu() 를 먼저 부릅니다 →", g.cpu().squeeze().shape)


## 오류 대응 요약

| 증상 | 원인 | 해결 |
| --- | --- | --- |
| mat1 and mat2 shapes cannot be multiplied | Flatten 누락 | 모델 첫 층에 `nn.Flatten()` |
| Target N is out of bounds | 출력 뉴런 수 부족 | 출력 = 클래스 수 |
| expected scalar type Long | 정답이 실수/one-hot | 정수 클래스 번호로 |
| Expected all tensors on the same device | 모델·데이터 장치 다름 | 둘 다 `.to(device)` |
| can't convert cuda tensor to numpy | GPU Tensor 시각화 | `.cpu()` 먼저 |

## 직접 해보기
아래 셀에는 오류가 두 개 있습니다. 메시지를 읽고 고치세요.


In [ ]:
m = nn.Sequential(nn.Linear(784, 10)).to(device)
x, y = next(iter(loader))
print(nn.CrossEntropyLoss()(m(x), y))
